In [1]:
"""
FinRL Tutorial - Part 1: Data Collection and Preprocessing (수정본)
Stock NeurIPS2018 시리즈 - 데이터 수집 및 전처리

이 파일은 Jupyter Notebook으로 변환하거나 직접 실행할 수 있습니다.
변환 명령: jupyter nbconvert --to notebook notebook_1_data_fixed.py
"""

'\nFinRL Tutorial - Part 1: Data Collection and Preprocessing (수정본)\nStock NeurIPS2018 시리즈 - 데이터 수집 및 전처리\n\n이 파일은 Jupyter Notebook으로 변환하거나 직접 실행할 수 있습니다.\n변환 명령: jupyter nbconvert --to notebook notebook_1_data_fixed.py\n'

# Stock NeurIPS2018 Part 1. Data (수정본)

이 노트북은 NeurIPS 2018 논문 "Practical Deep Reinforcement Learning Approach for Stock Trading"의
재현 및 개선 버전입니다.

**주요 수정 사항:**
- 최신 라이브러리 버전 호환성 개선
- 에러 핸들링 추가
- 진행 상황 로깅 추가
- 데이터 검증 강화

## Part 0. 실행 전 설정

이 섹션은 실행 환경을 확인하고 진행 상황을 추적합니다.

In [2]:
# 실행 로그 초기화
import json
import os
from datetime import datetime
from pathlib import Path

# 진행 상황 추적을 위한 로그 파일
PROGRESS_LOG_FILE = "PROGRESS_LOG.md"

def log_progress(phase, step, status, message=""):
    """진행 상황을 마크다운 파일에 기록"""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_entry = f"\n### [{timestamp}] Phase {phase}, Step {step}\n"
    log_entry += f"**Status**: {status}\n"
    if message:
        log_entry += f"**Message**: {message}\n"

    # 파일에 추가
    with open(PROGRESS_LOG_FILE, 'a', encoding='utf-8') as f:
        f.write(log_entry)

    # 콘솔에도 출력
    print(f"[{status}] Phase {phase}.{step}: {message}")

# 로그 파일 초기화
with open(PROGRESS_LOG_FILE, 'w', encoding='utf-8') as f:
    f.write("# FinRL Tutorial - 실행 진행 상황 로그\n")
    f.write(f"**시작 시간**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

log_progress(1, 0, "START", "데이터 수집 및 전처리 시작")

[START] Phase 1.0: 데이터 수집 및 전처리 시작


## Part 1. 패키지 설치 및 Import

필요한 패키지들을 설치하고 import합니다.

In [3]:
# 패키지 설치
import sys

log_progress(1, 1, "RUNNING", "필수 패키지 설치 중...")

# pip 업그레이드
!{sys.executable} -m pip install --upgrade pip -q

# 필수 패키지 설치 (특정 버전 명시)
packages_to_install = [
    "swig",
    "wrds>=3.1.5",
    "pyportfolioopt>=1.5.5",
    "yfinance>=0.2.28",
    "stable-baselines3>=2.0.0",
    "gymnasium>=0.28.1",
]

for package in packages_to_install:
    print(f"Installing {package}...")
    !{sys.executable} -m pip install {package} -q

# FinRL 설치 (최신 버전)
print("Installing FinRL...")
!{sys.executable} -m pip install git+https://github.com/AI4Finance-Foundation/FinRL.git -q

log_progress(1, 1, "COMPLETED", "패키지 설치 완료")

[RUNNING] Phase 1.1: 필수 패키지 설치 중...
Installing swig...
Installing wrds>=3.1.5...
Installing pyportfolioopt>=1.5.5...
Installing yfinance>=0.2.28...
Installing stable-baselines3>=2.0.0...
Installing gymnasium>=0.28.1...
Installing FinRL...
[COMPLETED] Phase 1.1: 패키지 설치 완료


In [4]:
# 패키지 Import 및 버전 확인
log_progress(1, 2, "RUNNING", "패키지 import 및 버전 확인 중...")

import pandas as pd
import numpy as np
import yfinance as yf
import itertools
import warnings
warnings.filterwarnings('ignore')

try:
    from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
    from finrl.meta.preprocessor.preprocessors import FeatureEngineer, data_split
    from finrl import config_tickers
    from finrl.config import INDICATORS

    print("FinRL import 성공!")
    log_progress(1, 2, "SUCCESS", "모든 패키지 import 완료")

except ImportError as e:
    error_msg = f"FinRL import 실패: {str(e)}"
    print(error_msg)
    log_progress(1, 2, "ERROR", error_msg)
    raise

# 버전 정보 출력
print("\n=== 패키지 버전 정보 ===")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"yfinance: {yf.__version__ if hasattr(yf, '__version__') else 'unknown'}")

[RUNNING] Phase 1.2: 패키지 import 및 버전 확인 중...
FinRL import 성공!
[SUCCESS] Phase 1.2: 모든 패키지 import 완료

=== 패키지 버전 정보 ===
pandas: 2.2.3
numpy: 2.2.6
yfinance: 0.2.58


## Part 2. 데이터 다운로드

Yahoo Finance에서 DJIA 30 종목의 주가 데이터를 다운로드합니다.

### 2.1 단일 종목 예제 (Apple Inc.)

In [5]:
log_progress(2, 1, "RUNNING", "Apple(AAPL) 데이터 다운로드 테스트...")

# yfinance 사용
try:
    aapl_df_yf = yf.download(tickers="aapl", start='2020-01-01', end='2020-01-31', progress=False)
    print("\n=== yfinance를 사용한 데이터 ===")
    print(aapl_df_yf.head())
    log_progress(2, 1, "SUCCESS", "yfinance 테스트 성공")
except Exception as e:
    log_progress(2, 1, "ERROR", f"yfinance 다운로드 실패: {str(e)}")

[RUNNING] Phase 2.1: Apple(AAPL) 데이터 다운로드 테스트...
YF.download() has changed argument auto_adjust default to True

=== yfinance를 사용한 데이터 ===
Price           Close       High        Low       Open     Volume
Ticker           AAPL       AAPL       AAPL       AAPL       AAPL
Date                                                             
2020-01-02  72.400505  72.460769  71.156667  71.409770  135480400
2020-01-03  71.696640  72.455958  71.472462  71.629145  146322800
2020-01-06  72.267929  72.306499  70.568503  70.819201  118387200
2020-01-07  71.928055  72.533095  71.708695  72.277578  108872000
2020-01-08  73.085098  73.386416  71.631544  71.631544  132079200
[SUCCESS] Phase 2.1: yfinance 테스트 성공


In [6]:
# FinRL YahooDownloader 사용
try:
    aapl_df_finrl = YahooDownloader(
        start_date='2020-01-01',
        end_date='2020-01-31',
        ticker_list=['aapl']
    ).fetch_data()

    print("\n=== FinRL YahooDownloader를 사용한 데이터 ===")
    print(aapl_df_finrl.head())
    log_progress(2, 1, "SUCCESS", "FinRL YahooDownloader 테스트 성공")
except Exception as e:
    log_progress(2, 1, "ERROR", f"FinRL 다운로드 실패: {str(e)}")
    raise

[*********************100%***********************]  1 of 1 completed

YF deprecation warning: set proxy via new config function: yf.set_config(proxy=proxy)
Shape of DataFrame:  (20, 8)

=== FinRL YahooDownloader를 사용한 데이터 ===
Price        date      close       high        low       open     volume  \
0      2020-01-02  72.400505  72.460769  71.156667  71.409770  135480400   
1      2020-01-03  71.696640  72.455958  71.472462  71.629145  146322800   
2      2020-01-06  72.267929  72.306499  70.568503  70.819201  118387200   
3      2020-01-07  71.928055  72.533095  71.708695  72.277578  108872000   
4      2020-01-08  73.085098  73.386416  71.631544  71.631544  132079200   

Price   tic  day  
0      aapl    3  
1      aapl    4  
2      aapl    0  
3      aapl    1  
4      aapl    2  
[SUCCESS] Phase 2.1: FinRL YahooDownloader 테스트 성공


### 2.2 DJIA 30 종목 데이터 다운로드

전체 30개 종목에 대해 학습 기간과 거래 기간의 데이터를 다운로드합니다.

In [7]:
# 티커 리스트 확인
print("=== DJIA 30 종목 리스트 ===")
print(config_tickers.DOW_30_TICKER)
print(f"\n총 {len(config_tickers.DOW_30_TICKER)}개 종목")

=== DJIA 30 종목 리스트 ===
['AXP', 'AMGN', 'AAPL', 'BA', 'CAT', 'CSCO', 'CVX', 'GS', 'HD', 'HON', 'IBM', 'INTC', 'JNJ', 'KO', 'JPM', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'PG', 'TRV', 'UNH', 'CRM', 'VZ', 'V', 'WBA', 'WMT', 'DIS', 'DOW']

총 30개 종목


In [8]:
# 날짜 범위 설정
TRAIN_START_DATE = '2009-01-01'
TRAIN_END_DATE = '2020-07-01'
TRADE_START_DATE = '2020-07-01'
TRADE_END_DATE = '2021-10-29'

print(f"\n=== 데이터 기간 설정 ===")
print(f"학습 기간: {TRAIN_START_DATE} ~ {TRAIN_END_DATE}")
print(f"거래 기간: {TRADE_START_DATE} ~ {TRADE_END_DATE}")


=== 데이터 기간 설정 ===
학습 기간: 2009-01-01 ~ 2020-07-01
거래 기간: 2020-07-01 ~ 2021-10-29


In [9]:
# 전체 데이터 다운로드 (재시도 로직 포함)
log_progress(2, 2, "RUNNING", "DJIA 30 종목 데이터 다운로드 중... (시간이 걸릴 수 있습니다)")

max_retries = 3
retry_count = 0
df_raw = None

while retry_count < max_retries:
    try:
        df_raw = YahooDownloader(
            start_date=TRAIN_START_DATE,
            end_date=TRADE_END_DATE,
            ticker_list=config_tickers.DOW_30_TICKER
        ).fetch_data()

        # 데이터 검증
        if df_raw is not None and len(df_raw) > 0:
            print(f"\n다운로드 성공! 총 {len(df_raw)}개 행")
            print(f"기간: {df_raw['date'].min()} ~ {df_raw['date'].max()}")
            print(f"종목 수: {df_raw['tic'].nunique()}")
            log_progress(2, 2, "SUCCESS", f"데이터 다운로드 완료 ({len(df_raw)}행)")
            break
        else:
            raise ValueError("다운로드된 데이터가 비어있습니다")

    except Exception as e:
        retry_count += 1
        error_msg = f"다운로드 시도 {retry_count}/{max_retries} 실패: {str(e)}"
        print(error_msg)
        log_progress(2, 2, "RETRY", error_msg)

        if retry_count >= max_retries:
            log_progress(2, 2, "ERROR", "최대 재시도 횟수 초과")
            raise Exception(f"데이터 다운로드 실패 (재시도 {max_retries}회)")

[RUNNING] Phase 2.2: DJIA 30 종목 데이터 다운로드 중... (시간이 걸릴 수 있습니다)


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

Shape of DataFrame:  (91072, 8)

다운로드 성공! 총 91072개 행
기간: 2009-01-02 ~ 2021-10-28
종목 수: 29
[SUCCESS] Phase 2.2: 데이터 다운로드 완료 (91072행)


In [10]:
# 다운로드된 데이터 확인
print("\n=== 다운로드된 데이터 샘플 ===")
print(df_raw.head())

print("\n=== 데이터 정보 ===")
print(df_raw.info())

print("\n=== 결측치 확인 ===")
print(df_raw.isnull().sum())


=== 다운로드된 데이터 샘플 ===
Price        date      close       high        low       open     volume  \
0      2009-01-02   2.719142   2.727832   2.551650   2.573223  746015200   
1      2009-01-02  39.900173  39.961048  39.061450  39.629617    6547900   
2      2009-01-02  14.821131  14.966812  14.108060  14.238406   10955700   
3      2009-01-02  33.941090  34.173615  32.088393  32.103395    7010200   
4      2009-01-02  30.076950  30.121831  28.666391  28.794624    7117200   

Price   tic  day  
0      AAPL    4  
1      AMGN    4  
2       AXP    4  
3        BA    4  
4       CAT    4  

=== 데이터 정보 ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 91072 entries, 0 to 91071
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   date    91072 non-null  object 
 1   close   91072 non-null  float64
 2   high    91072 non-null  float64
 3   low     91072 non-null  float64
 4   open    91072 non-null  float64
 5   volume  91072 non-n

## Part 3. 데이터 전처리

기술적 지표(Technical Indicators)를 추가하고 데이터를 전처리합니다.

In [11]:
log_progress(3, 1, "RUNNING", "기술적 지표 추가 중...")

# FeatureEngineer 초기화
try:
    fe = FeatureEngineer(
        use_technical_indicator=True,
        tech_indicator_list=INDICATORS,
        use_vix=True,
        use_turbulence=True,
        user_defined_feature=False
    )

    # 데이터 전처리 실행
    processed = fe.preprocess_data(df_raw)

    print(f"\n전처리 완료! 처리된 데이터: {len(processed)}행")
    print(f"추가된 기술적 지표: {INDICATORS}")
    log_progress(3, 1, "SUCCESS", f"기술적 지표 추가 완료 ({len(INDICATORS)}개)")

except Exception as e:
    error_msg = f"전처리 실패: {str(e)}"
    print(error_msg)
    log_progress(3, 1, "ERROR", error_msg)
    raise

[RUNNING] Phase 3.1: 기술적 지표 추가 중...
Successfully added technical indicators


[*********************100%***********************]  1 of 1 completed


Shape of DataFrame:  (3228, 8)
Successfully added vix
Successfully added turbulence index

전처리 완료! 처리된 데이터: 90384행
추가된 기술적 지표: ['macd', 'boll_ub', 'boll_lb', 'rsi_30', 'cci_30', 'dx_30', 'close_30_sma', 'close_60_sma']
[SUCCESS] Phase 3.1: 기술적 지표 추가 완료 (8개)


In [12]:
# 전처리된 데이터 확인
print("\n=== 전처리된 데이터 샘플 ===")
print(processed.head())

print("\n=== 새로 추가된 컬럼들 ===")
new_columns = [col for col in processed.columns if col not in df_raw.columns]
print(new_columns)


=== 전처리된 데이터 샘플 ===
         date      close       high        low       open     volume   tic  \
0  2009-01-02   2.719142   2.727832   2.551650   2.573223  746015200  AAPL   
1  2009-01-02  39.900173  39.961048  39.061450  39.629617    6547900  AMGN   
2  2009-01-02  14.821131  14.966812  14.108060  14.238406   10955700   AXP   
3  2009-01-02  33.941090  34.173615  32.088393  32.103395    7010200    BA   
4  2009-01-02  30.076950  30.121831  28.666391  28.794624    7117200   CAT   

   day  macd   boll_ub   boll_lb  rsi_30     cci_30  dx_30  close_30_sma  \
0    4   0.0  2.938815  2.614228   100.0  66.666667  100.0      2.719142   
1    4   0.0  2.938815  2.614228   100.0  66.666667  100.0     39.900173   
2    4   0.0  2.938815  2.614228   100.0  66.666667  100.0     14.821131   
3    4   0.0  2.938815  2.614228   100.0  66.666667  100.0     33.941090   
4    4   0.0  2.938815  2.614228   100.0  66.666667  100.0     30.076950   

   close_60_sma        vix  turbulence  
0      2.719

In [13]:
# 데이터 정규화 및 결측치 처리
log_progress(3, 2, "RUNNING", "데이터 정규화 및 결측치 처리 중...")

# 모든 날짜-티커 조합 생성 (결측 데이터 보완)
list_ticker = processed["tic"].unique().tolist()
list_date = list(pd.date_range(processed['date'].min(), processed['date'].max()).astype(str))
combination = list(itertools.product(list_date, list_ticker))

processed_full = pd.DataFrame(combination, columns=["date", "tic"]).merge(
    processed, on=["date", "tic"], how="left"
)

# 실제 거래일만 필터링
processed_full = processed_full[processed_full['date'].isin(processed['date'])]
processed_full = processed_full.sort_values(['date', 'tic'])

# 결측치를 0으로 채우기
processed_full = processed_full.fillna(0)

print(f"\n정규화 완료! 최종 데이터: {len(processed_full)}행")
log_progress(3, 2, "SUCCESS", "데이터 정규화 및 결측치 처리 완료")

[RUNNING] Phase 3.2: 데이터 정규화 및 결측치 처리 중...

정규화 완료! 최종 데이터: 90384행
[SUCCESS] Phase 3.2: 데이터 정규화 및 결측치 처리 완료


In [14]:
# 최종 데이터 확인
print("\n=== 최종 데이터 샘플 ===")
print(processed_full.head(10))

print("\n=== 결측치 재확인 ===")
print(f"결측치 총 개수: {processed_full.isnull().sum().sum()}")


=== 최종 데이터 샘플 ===
         date   tic      close       high        low       open       volume  \
0  2009-01-02  AAPL   2.719142   2.727832   2.551650   2.573223  746015200.0   
1  2009-01-02  AMGN  39.900173  39.961048  39.061450  39.629617    6547900.0   
2  2009-01-02   AXP  14.821131  14.966812  14.108060  14.238406   10955700.0   
3  2009-01-02    BA  33.941090  34.173615  32.088393  32.103395    7010200.0   
4  2009-01-02   CAT  30.076950  30.121831  28.666391  28.794624    7117200.0   
5  2009-01-02   CRM   8.402902   8.447362   7.817514   7.928663    4069200.0   
6  2009-01-02  CSCO  10.973989  10.999872  10.514584  10.618112   40980600.0   
7  2009-01-02   CVX  38.469288  38.861424  36.991249  37.318028   13695900.0   
8  2009-01-02   DIS  20.123596  20.216138  18.928968  19.147703    9796600.0   
9  2009-01-02    GS  64.347267  64.985104  60.957837  62.315088   14088500.0   

   day  macd   boll_ub   boll_lb  rsi_30     cci_30  dx_30  close_30_sma  \
0  4.0   0.0  2.938815  

## Part 4. 데이터 분할 및 저장

학습용(train)과 거래용(trade) 데이터로 분할하고 CSV 파일로 저장합니다.

In [15]:
log_progress(4, 1, "RUNNING", "데이터 분할 중...")

# 데이터 분할
try:
    train = data_split(processed_full, TRAIN_START_DATE, TRAIN_END_DATE)
    trade = data_split(processed_full, TRADE_START_DATE, TRADE_END_DATE)

    print(f"\n=== 데이터 분할 결과 ===")
    print(f"학습 데이터: {len(train)}행")
    print(f"거래 데이터: {len(trade)}행")
    print(f"학습 기간: {train['date'].min()} ~ {train['date'].max()}")
    print(f"거래 기간: {trade['date'].min()} ~ {trade['date'].max()}")

    log_progress(4, 1, "SUCCESS", f"데이터 분할 완료 (train: {len(train)}, trade: {len(trade)})")

except Exception as e:
    error_msg = f"데이터 분할 실패: {str(e)}"
    print(error_msg)
    log_progress(4, 1, "ERROR", error_msg)
    raise

[RUNNING] Phase 4.1: 데이터 분할 중...

=== 데이터 분할 결과 ===
학습 데이터: 81004행
거래 데이터: 9380행
학습 기간: 2009-01-02 ~ 2020-06-30
거래 기간: 2020-07-01 ~ 2021-10-27
[SUCCESS] Phase 4.1: 데이터 분할 완료 (train: 81004, trade: 9380)


In [16]:
# 데이터 저장
log_progress(4, 2, "RUNNING", "데이터를 CSV 파일로 저장 중...")

try:
    train.to_csv('train_data.csv', index=False)
    trade.to_csv('trade_data.csv', index=False)

    # 파일 크기 확인
    train_size = os.path.getsize('train_data.csv') / (1024 * 1024)  # MB
    trade_size = os.path.getsize('trade_data.csv') / (1024 * 1024)  # MB

    print(f"\n=== 파일 저장 완료 ===")
    print(f"train_data.csv: {train_size:.2f} MB")
    print(f"trade_data.csv: {trade_size:.2f} MB")

    log_progress(4, 2, "SUCCESS", f"CSV 파일 저장 완료 (train: {train_size:.2f}MB, trade: {trade_size:.2f}MB)")

except Exception as e:
    error_msg = f"파일 저장 실패: {str(e)}"
    print(error_msg)
    log_progress(4, 2, "ERROR", error_msg)
    raise

[RUNNING] Phase 4.2: 데이터를 CSV 파일로 저장 중...

=== 파일 저장 완료 ===
train_data.csv: 22.14 MB
trade_data.csv: 2.58 MB
[SUCCESS] Phase 4.2: CSV 파일 저장 완료 (train: 22.14MB, trade: 2.58MB)


## Part 5. 데이터 검증

저장된 데이터를 다시 로드하여 무결성을 검증합니다.

In [ ]:
log_progress(5, 1, "RUNNING", "저장된 데이터 검증 중...")

try:
    # 파일 다시 로드
    train_loaded = pd.read_csv('train_data.csv')
    trade_loaded = pd.read_csv('trade_data.csv')

    print("\n=== 데이터 검증 결과 ===")

    # 크기 확인
    assert len(train_loaded) == len(train), "학습 데이터 크기 불일치"
    assert len(trade_loaded) == len(trade), "거래 데이터 크기 불일치"
    print("✓ 데이터 크기 일치")

    # 컬럼 확인
    assert list(train_loaded.columns) == list(train.columns), "학습 데이터 컬럼 불일치"
    assert list(trade_loaded.columns) == list(trade.columns), "거래 데이터 컬럼 불일치"
    print("✓ 컬럼 구조 일치")

    # 결측치 확인
    assert train_loaded.isnull().sum().sum() == 0, "학습 데이터에 결측치 존재"
    assert trade_loaded.isnull().sum().sum() == 0, "거래 데이터에 결측치 존재"
    print("✓ 결측치 없음")

    print("\n모든 검증 통과!")
    log_progress(5, 1, "SUCCESS", "데이터 검증 완료 - 모든 테스트 통과")

except AssertionError as e:
    error_msg = f"데이터 검증 실패: {str(e)}"
    print(error_msg)
    log_progress(5, 1, "ERROR", error_msg)
    raise

except Exception as e:
    error_msg = f"데이터 검증 중 오류: {str(e)}"
    print(error_msg)
    log_progress(5, 1, "ERROR", error_msg)
    raise

## 완료!

데이터 수집 및 전처리가 완료되었습니다.

**생성된 파일:**
- `train_data.csv`: 학습용 데이터
- `trade_data.csv`: 거래용(백테스팅) 데이터
- `PROGRESS_LOG.md`: 실행 진행 상황 로그

**다음 단계:**
`Stock_NeurIPS2018_2_Train_Fixed.ipynb`를 실행하여 DRL 모델을 학습하세요.

In [ ]:
# 최종 요약
log_progress(0, 0, "COMPLETED", "데이터 수집 및 전처리 완료")

print("\n" + "="*60)
print("데이터 준비 완료!")
print("="*60)
print(f"\n생성된 파일:")
print(f"  - train_data.csv ({len(train)}행)")
print(f"  - trade_data.csv ({len(trade)}행)")
print(f"  - PROGRESS_LOG.md (실행 로그)")
print(f"\n다음 단계: Stock_NeurIPS2018_2_Train_Fixed.ipynb 실행")
print("="*60)